# Experiment [4] Visual Dashboard: Strict Dataset-Heldout JEPA

Strict protocol: each target dataset is excluded from SSL pretraining, excluded from probe training, then used only for testing. This is the cleanest domain-transfer evidence in the current line.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "[4]"
DATA = ROOT / "data"

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "raw_flattened": "#64748b",
    "raw_stats": "#f97316",
    "surf_jepa_v0_plus_raw_stats": "#16a34a",
    "large_dual_s2": "#2563eb",
    "large_default": "#7c3aed",
    "large_dual_s2_jepa": "#2563eb",
    "large_default_jepa": "#7c3aed",
    "presto": "#db2777",
    "olmoearth": "#059669",
}
PRIORITY_CONDITIONS = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
HOLDOUT_ORDER = ["rwanda-ceo", "togo", "togo-eval", "ethiopia", "lem-brazil"]

pd.options.display.max_columns = 80
pd.options.display.max_rows = 80


def standardize_table(df):
    if df.empty:
        return df
    rename = {
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "mean_auc": "auc",
        "mean_f1": "f1",
        "budget": "label_budget",
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns and v not in df.columns})


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT) if path.is_absolute() and ROOT in path.parents else path}")
        return pd.DataFrame()
    return standardize_table(pd.read_csv(path, **kwargs))


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def pct(x):
    return f"{100*x:.1f}%" if pd.notna(x) else ""


def scorecard(df, cols, title="Scorecard", by=None):
    if df.empty:
        print("No data for scorecard")
        return
    work = df.copy()
    if by:
        work = work.set_index(by)
    display(work[cols].style.format({c: "{:.4f}" for c in cols if pd.api.types.is_numeric_dtype(work[c])}).set_caption(title))


def bar_metric(df, x, y, color=None, title=None, barmode="group", text=True, height=460):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode=barmode,
        text_auto=".3f" if text else False,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="", xaxis_title="", yaxis_title=y)
    fig.show()
    return fig


def line_metric(df, x, y, color=None, facet_col=None, title=None, markers=True, height=520):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.line(
        df,
        x=x,
        y=y,
        color=color,
        facet_col=facet_col,
        markers=markers,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="")
    fig.show()
    return fig


def heatmap(df, x, y, z, title=None, height=520, color_continuous_scale="RdYlGn"):
    if df.empty:
        print("No data to plot")
        return None
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig


def priority_average(df, group_cols, metric_cols=("f1", "auc", "balanced_accuracy"), conditions=PRIORITY_CONDITIONS):
    if df.empty:
        return df
    work = df[df["condition"].isin(conditions)].copy() if "condition" in df else df.copy()
    work = work.rename(columns={"balanced_acc": "balanced_accuracy", "mean_auc": "auc", "mean_f1": "f1"})
    available = [c for c in metric_cols if c in work.columns]
    if not available:
        return work.groupby(group_cols, dropna=False).size().reset_index(name="n")
    return work.groupby(group_cols, dropna=False)[available].mean().reset_index()


def normalize_model_name(name):
    text = str(name)
    return text.replace("_seed42", "").replace("_seed7", "").replace("_seed11", "")

RUN_DIR = ARTIFACTS / "repair_4_strict"
ANALYSIS = RUN_DIR / "analysis"

In [2]:
ranking = read_csv(ANALYSIS / "strict_priority_ranking.csv")
scorecard(ranking, [c for c in ranking.columns if c not in {"model"}], "[4] Strict priority ranking", by="model" if "model" in ranking else None)
if not ranking.empty:
    model_col = "model" if "model" in ranking else "config"
    metric = "priority_f1" if "priority_f1" in ranking else "f1"
    bar_metric(ranking.sort_values(metric), x=model_col, y=metric, color=model_col, title="[4] Strict priority F1")

,config,n,priority_f1,priority_f1_sd,priority_auc,priority_bal_acc
model,,,,,,
raw_flattened,large_default,15.0000,0.3112,0.0405,0.6111,0.5437
raw_flattened,large_dual_s2,15.0000,0.3112,0.0405,0.6111,0.5437
surf_jepa_v0,large_dual_s2,15.0000,0.4941,0.1223,0.6703,0.6093
surf_jepa_v0,large_default,15.0000,0.4898,0.1223,0.6714,0.6088


In [3]:
probe = read_csv(ANALYSIS / "strict_probe_results_all.csv")
print(probe.shape)
probe.head()

(1500, 10)


,config,holdout,seed,model,condition,label_budget,f1,auc,balanced_accuracy,n_test
0,large_default,ethiopia,11,surf_jepa_v0,clean,0.01,0.717797,0.716339,0.638685,830
1,large_default,ethiopia,11,surf_jepa_v0,clean,0.05,0.632111,0.743591,0.659639,830
2,large_default,ethiopia,11,surf_jepa_v0,clean,0.10,0.658260,0.778090,0.683722,830
3,large_default,ethiopia,11,surf_jepa_v0,clean,0.25,0.701754,0.798019,0.723264,830
4,large_default,ethiopia,11,surf_jepa_v0,clean,1.00,0.724566,0.826360,0.741799,830


In [4]:
if not probe.empty:
    model_col = "model" if "model" in probe else "config"
    priority = priority_average(probe, [model_col, "holdout"])
    heatmap(priority, x="holdout", y=model_col, z="f1", title="[4] Strict F1 by heldout", height=560)
    heatmap(priority, x="holdout", y=model_col, z="auc", title="[4] Strict AUROC by heldout", height=560, color_continuous_scale="Viridis")

In [5]:
cond = read_csv(ANALYSIS / "strict_condition_summary.csv")
if not cond.empty:
    model_col = "model" if "model" in cond else "config"
    metric = "f1" if "f1" in cond else "mean_f1"
    heatmap(cond, x="condition", y=model_col, z=metric, title="[4] Strict stress condition F1", height=560)

In [6]:
raw_delta = read_csv(ANALYSIS / "strict_jepa_minus_raw_by_condition.csv")
if not raw_delta.empty:
    display(raw_delta)
    delta_col = "delta_f1" if "delta_f1" in raw_delta else [c for c in raw_delta.columns if "f1" in c.lower()][-1]
    bar_metric(raw_delta, x="condition", y=delta_col, color="model" if "model" in raw_delta else None, title="[4] JEPA minus raw by condition", text=False)

,config,model_jepa,condition,f1_jepa,auc_jepa,bal_acc_jepa,model_raw,f1_raw,auc_raw,bal_acc_raw,delta_f1,delta_auc,delta_bal_acc
0,large_default,surf_jepa_v0,clean,0.509398,0.690445,0.624336,raw_flattened,0.269178,0.644351,0.542690,0.240220,0.046094,0.081646
1,large_default,surf_jepa_v0,s2_off_tdrop50,0.455340,0.640184,0.587924,raw_flattened,0.358572,0.578440,0.536974,0.096768,0.061744,0.050950
2,large_default,surf_jepa_v0,sensor_off_s2,0.492400,0.707178,0.623224,raw_flattened,0.227366,0.632833,0.526342,0.265034,0.074344,0.096882
3,large_default,surf_jepa_v0,temporal_drop_50,0.497980,0.667682,0.609135,raw_flattened,0.344679,0.604703,0.554272,0.153302,0.062979,0.054863
4,large_default,surf_jepa_v0,temporal_drop_70,0.493773,0.651314,0.599581,raw_flattened,0.355977,0.595215,0.558420,0.137796,0.056099,0.041161
5,large_dual_s2,surf_jepa_v0,clean,0.509575,0.689662,0.623046,raw_flattened,0.269178,0.644351,0.542690,0.240397,0.045311,0.080356
6,large_dual_s2,surf_jepa_v0,s2_off_tdrop50,0.459642,0.638703,0.587985,raw_flattened,0.358572,0.578440,0.536974,0.101070,0.060263,0.051012
7,large_dual_s2,surf_jepa_v0,sensor_off_s2,0.504896,0.701497,0.624664,raw_flattened,0.227366,0.632833,0.526342,0.277530,0.068663,0.098322
8,large_dual_s2,surf_jepa_v0,temporal_drop_50,0.499893,0.668789,0.610434,raw_flattened,0.344679,0.604703,0.554272,0.155214,0.064086,0.056162
9,large_dual_s2,surf_jepa_v0,temporal_drop_70,0.496312,0.652639,0.600557,raw_flattened,0.355977,0.595215,0.558420,0.140334,0.057424,0.042137


In [7]:
if not probe.empty:
    model_col = "model" if "model" in probe else "config"
    curves = probe[probe["condition"].isin(PRIORITY_CONDITIONS)].groupby([model_col, "holdout", "label_budget"])[["f1", "auc"]].mean().reset_index()
    line_metric(curves, x="label_budget", y="f1", color=model_col, facet_col="holdout", title="[4] Strict heldout label-efficiency", height=740)